# Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, sys
for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == 'Multifirefly-Project':
        os.chdir(p)
        sys.path.insert(0, str(p / 'multiff_analysis/multiff_code/methods'))
        break

import sys


from data_wrangling import general_utils, specific_utils, process_monkey_information, base_processing_class, retrieve_raw_data, further_processing_class
from pattern_discovery import pattern_by_trials, pattern_by_points, make_ff_dataframe, ff_dataframe_utils, pattern_by_trials, pattern_by_points, cluster_analysis, organize_patterns_and_features, category_class
from decision_making_analysis.ff_data_acquisition import cluster_replacement_utils
from decision_making_analysis.ff_data_acquisition import ff_data_utils
from decision_making_analysis.data_compilation import miss_events_across_sessions
from decision_making_analysis.ff_data_acquisition import get_missed_ff_data

from decision_making_analysis. ff_data_acquisition import free_selection, cluster_replacement_utils
from visualization.matplotlib_tools import plot_trials, plot_polar, additional_plots, plot_behaviors_utils, plot_statistics, monkey_heading_utils
from visualization.animation import animation_func, animation_utils, animation_class
from machine_learning.ml_methods import regression_utils, classification_utils, prep_ml_data_utils, hyperparam_tuning_class
from reinforcement_learning.base_classes import env_utils, base_env, more_envs, rl_base_class, rl_base_utils
from reinforcement_learning.agents.rnn import gru_utils, lstm_utils
from reinforcement_learning.agents.feedforward import interpret_neural_network, sb3_class, sb3_utils
from null_behaviors import show_null_trajectory, find_best_arc, curvature_utils
from decision_making_analysis.data_compilation import miss_events_class
from decision_making_analysis.ff_data_acquisition import ff_data_utils
from reinforcement_learning.collect_data import agent_patterns_class

import os, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from os.path import exists
import seaborn as sns
import math
import copy
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from scipy import stats
from IPython.display import HTML
from matplotlib import rc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, hamming_loss, multilabel_confusion_matrix, fbeta_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier
from sklearn.neural_network import MLPClassifier
from math import pi
from scipy.ndimage import gaussian_filter1d
from importlib import reload

plt.rcParams["animation.html"] = "html5"
os.environ['KMP_DUPLICATE_LIB_OK']='True'
rc('animation', html='jshtml')
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams['animation.embed_limit'] = 2**128
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True)
pd.options.display.max_rows = 101


# Get data

## agent

In [ ]:
model_folder_name = 'RL_models/sb3_stored_models/all_agents/agents_with_noise/ff5_mem2p5_drop_fill_visible_only/vn0_wn0_pr0p0005_pt0p005_mr0p002_mt0p002'
ap = agent_patterns_class.AgentPatterns(model_folder_name=model_folder_name)
ap.combine_or_retrieve_patterns_and_features()

## monkey

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0314"
data_item = animation_class.AnimationClass(raw_data_folder_path=raw_data_folder_path)
data_item.retrieve_or_make_monkey_data()
data_item.make_or_retrieve_ff_dataframe(exists_ok=True)

data_item.find_patterns()
data_item.make_PlotTrials_args()
data_item.make_df_related_to_patterns_and_features()

## organize data

In [ ]:
# The original intention of the code is that pattern_frequencies_a and feature_statistics_a will be freshly generated
# But now, for the sake of demonstration...
pattern_frequencies_a = ap.combd_pattern_frequencies.copy()
pattern_frequencies_m = data_item.pattern_frequencies.copy()
feature_statistics_a = ap.combd_feature_statistics.copy()
feature_statistics_m = data_item.feature_statistics.copy()
pattern_frequencies_a['Player'] = "Agent"
feature_statistics_a['Player'] = "Agent"

In [ ]:
agent_monkey_pattern_frequencies = organize_patterns_and_features.combine_df_of_agent_and_monkey(pattern_frequencies_m, pattern_frequencies_a, agent_names = ["Agent", "Agent2", "Agent3"])
agent_monkey_feature_statistics = organize_patterns_and_features.combine_df_of_agent_and_monkey(feature_statistics_m, feature_statistics_a, agent_names = ["Agent", "Agent2", "Agent3"]) 

# Plot statistics

## Plotting: same plot

In [ ]:
plot_statistics.plot_merged_df(agent_monkey_pattern_frequencies, x="label", y="rate")
plt.show()

temp_agent_monkey_feature_statistics = agent_monkey_feature_statistics.copy()
temp_agent_monkey_feature_statistics = temp_agent_monkey_feature_statistics[temp_agent_monkey_feature_statistics['label']!='distance target last seen']
plot_statistics.plot_merged_df(temp_agent_monkey_feature_statistics, x="label", y="median")
plt.show()

## Plotting: individual plots

In [ ]:
reload(plot_statistics)

In [ ]:
plot_statistics.plot_merged_df_by_category(agent_monkey_feature_statistics, category_column_name="label for median", y="median", category_order=None, percentage=False)
plt.show()

## Plotting histograms

In [ ]:
data_item.make_or_retrieve_all_trial_patterns(exists_ok=True)
data_item.make_or_retrieve_pattern_frequencies(exists_ok=True)
data_item.make_or_retrieve_all_trial_features(exists_ok=True)
data_item.make_or_retrieve_feature_statistics(exists_ok=True)

In [ ]:
# all_trial_features_m means all_trial_features for monkey
all_trial_features_m = data_item.all_trial_features.copy()
# all_trial_features_m = pd.read_csv(data_folder_name + '/patterns_and_features/all_trial_features.csv')
all_trial_features_valid_m = all_trial_features_m[(all_trial_features_m['t_last_vis']<50) & (all_trial_features_m['hitting_arena_edge']==False)].reset_index()
median_values_m = all_trial_features_valid_m.median(axis=0)
all_trial_features_valid_m = all_trial_features_m[(all_trial_features_m['t_last_vis'] < 50) & (all_trial_features_m['hitting_arena_edge']==False)].reset_index()

In [ ]:
all_trial_features_valid = ap.combd_all_trial_features[(ap.combd_all_trial_features['t_last_vis'] < 50) & (ap.combd_all_trial_features['hitting_arena_edge']==False)].reset_index()
plot_statistics.plot_feature_histograms_for_monkey_and_agent(all_trial_features_valid_m, all_trial_features_valid)